# 01 — Home Credit: análise exploratória
Projeto 02 de Fundamentos de IA — Grupo 09.

**Estado:** notebook preparado; resultados reais ainda por executar e interpretar.
A identificação do grupo está no README. As contribuições individuais serão preenchidas
de acordo com o trabalho efetivamente realizado.

Fonte: [Home Credit Default Risk](https://www.kaggle.com/competitions/home-credit-default-risk/data).

Perguntas propostas:
1. Que características permitem prever dificuldades de pagamento (TARGET)?
2. Que perfis financeiros de pedidos podemos descobrir sem usar TARGET?

O dataset não é especificamente português e não garante crédito à habitação.
pandas autorizado provisoriamente pelo grupo. Confirmar o enquadramento temático e as bibliotecas auxiliares com o docente.

## Dados e execução
Executar a partir da raiz do repositório, depois de instalar requirements.txt.
Colocar application_train.csv e HomeCredit_columns_description.csv em data/home_credit,
ou definir HOME_CREDIT_DATA para a pasta que contém os ficheiros.

Se a conta Kaggle estiver autenticada e as regras já tiverem sido aceites pelo utilizador,
DOWNLOAD=True permite obter apenas os dois CSV. Nunca colocar tokens no notebook.

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from libs.data import FEATURES, load_applications

SEED = 42
# Mantém downloads explícitos: a autenticação e as regras são geridas no Kaggle.
DOWNLOAD = False
applications, dictionary = load_applications(download=DOWNLOAD)
print(f"Pedidos: {len(applications):,}; colunas: {len(applications.columns)}")

# Guardar estas informações com os resultados permite repetir a experiência.
display(pd.DataFrame(applications.attrs["source_files"]))
print({"Python": platform.python_version(), "pandas": pd.__version__, "NumPy": np.__version__})

## Metadados e qualidade
Consultar as definições originais antes de interpretar variáveis ou tratar anomalias.

In [ ]:
display(dictionary)
display(applications.head())
quality = pd.DataFrame({
    "tipo": applications.dtypes.astype(str),
    "em_falta": applications.isna().sum(),
    "percentagem_em_falta": applications.isna().mean() * 100,
    "valores_distintos": applications.nunique(),
})
display(quality.sort_values("percentagem_em_falta", ascending=False))
print("Duplicados de linha:", applications.duplicated().sum())
display(applications.select_dtypes(include="number").describe(percentiles=[0.01, 0.5, 0.99]).T)

## Variável-alvo e desequilíbrio
Confirmar a definição exata de TARGET no dicionário. Não interpretar TARGET=0 como
garantia de que todos os pagamentos foram sempre pontuais.
A accuracy de prever a classe maioritária é apenas uma referência descritiva nesta EDA.

In [ ]:
counts = applications["TARGET"].value_counts().sort_index()
display(pd.DataFrame({"pedidos": counts, "percentagem": counts / counts.sum() * 100}))
counts.plot.bar(xlabel="TARGET", ylabel="Pedidos", title="Distribuição das classes", rot=0)
plt.tight_layout()
plt.show()
print(f"Referência maioritária na base completa: {counts.max() / counts.sum():.2%}")

## Seleção inicial de variáveis
A primeira experiência usa sete variáveis numéricas financeiras e do agregado.
É uma proposta interpretável, não uma seleção otimizada nem um resultado.

Variáveis categóricas e fontes externas de scoring ficam fora desta primeira versão.
Avaliar na EDA se esta restrição prejudica a classificação e se há redundância no clustering.

In [ ]:
display(quality.loc[FEATURES])
applications[FEATURES].hist(bins=40, figsize=(12, 10))
plt.tight_layout()
plt.show()
display(applications[FEATURES].corr())

## Registo de decisões — preencher após executar
- Significado de uma linha e de TARGET, segundo o dicionário.
- Problemas de qualidade confirmados, incluindo valores negativos e extremos.
- Justificação de cada variável e de eventuais exclusões.
- Adequação das transformações propostas: medianas, log1p dos montantes e normalização.
- Limitações de representar perfis com estas variáveis.
- Tempo de execução e adequação do limite inicial de 20 000 pedidos.

Não escolher variáveis ou ajustes pelo desempenho no teste.
Nos notebooks seguintes, medianas e escalas da classificação são aprendidas apenas no treino.